# Предобработка OCR паспорта РФ

Первый этап: загрузка исходных данных, нормализация текста, разбор MRZ, восстановление доступных реквизитов, проверки дат и извлечение признаков. Затем выполняются 22 теста этих функций.

Запуск: `Restart Kernel` → `Run All`. Нужны исходные JSONL из `data/` и справочник `reference/subdivisions.csv`. Обученная модель и результаты предыдущих запусков не используются.

Следующий этап — [analysis.ipynb](analysis.ipynb): анализ ошибок, обучение, выбор порога и применение полученной модели.


In [116]:
import csv
import json
import math
import re
from datetime import date, timedelta
from pathlib import Path
from statistics import mean

### Нормализация и форматы


In [117]:
FIELDS = (
    "Last_name_ru",
    "First_name_ru",
    "Middle_name_ru",
    "Sex_ru",
    "Birth_date",
    "Birth_place_ru",
    "Issue_date",
    "Issue_organisation_code",
    "Licence_number",
    "Issue_organization_ru",
)


NAME_FIELDS = FIELDS[:3]
NUMERIC_FIELDS = {"Birth_date", "Issue_date", "Issue_organisation_code", "Licence_number"}
CYRILLIC_LOOKALIKES = str.maketrans(dict(zip("ABCEHKMOPTXY", "АВСЕНКМОРТХУ")))
DIGIT_LOOKALIKES = str.maketrans({"О": "0", "O": "0", "З": "3", "В": "8", "Б": "6"})


def canonical(value):
    """Exactly the comparison convention used by the supplied evaluate.py."""
    return " ".join(value.upper().split())


def field_text(value):
    return " ".join(line["text"] for line in value["lines"]) if value else ""


def normalize(field, value):
    text = canonical(value)
    if field in NUMERIC_FIELDS:
        text = text.translate(DIGIT_LOOKALIKES)
        if field in {"Birth_date", "Issue_date"}:
            match = re.fullmatch(r"([0-9]{2})[./-]([0-9]{2})[./-]([0-9]{4})", text)
            return ".".join(match.groups()) if match else text
        if field == "Issue_organisation_code":
            digits = re.sub(r"[\s-]", "", text)
            return digits[:3] + "-" + digits[3:] if re.fullmatch(r"[0-9]{6}", digits) else text
        digits = re.sub(r"\s", "", text)
        return (
            f"{digits[:2]} {digits[2:4]} {digits[4:]}"
            if re.fullmatch(r"[0-9]{10}", digits)
            else text
        )
    if field in NAME_FIELDS or field == "Sex_ru":
        text = text.translate(CYRILLIC_LOOKALIKES)
    else:
        # Foreign place names occur in the labels. Correct lookalikes only in
        # tokens dominated by Cyrillic, preserving fully Latin words and N20.
        def fix_token(match):
            token = match.group()
            cyrillic = len(re.findall(r"[А-ЯЁ]", token))
            latin = len(re.findall(r"[A-Z]", token))
            return token.translate(CYRILLIC_LOOKALIKES) if cyrillic > latin else token

        text = re.sub(r"[А-ЯЁA-Z]+", fix_token, text)
    if field == "Sex_ru" and text in {"МУЖ", "ЖЕН"}:
        return text + "."
    return text


def valid_format(field, text):
    if field in NAME_FIELDS:
        return bool(re.fullmatch(r"[А-ЯЁ]+(?:[ -][А-ЯЁ]+)*", text)) or (
            field == "Middle_name_ru" and text == ""
        )
    if field in {"Birth_date", "Issue_date"}:
        return bool(re.fullmatch(r"[0-9]{2}\.[0-9]{2}\.[0-9]{4}", text))
    if field == "Sex_ru":
        return text in {"МУЖ.", "ЖЕН."}
    if field == "Licence_number":
        return bool(re.fullmatch(r"[0-9]{2} [0-9]{2} [0-9]{6}", text))
    if field == "Issue_organisation_code":
        return bool(re.fullmatch(r"[0-9]{3}-[0-9]{3}", text))
    return bool(text) and bool(re.fullmatch(r"""[А-ЯЁA-Z0-9\s.,'’"№()/-]+""", text))

### Машиночитаемая зона


In [118]:
RUSSIAN = "АБВГДЕЁЖЗИЙКЛМНОПРСТУФХЦЧШЩЪЫЬЭЮЯ"
CLER = "ABVGDE2JZIQKLMNOPRSTUFHC34WXY9678"
ENCODE = str.maketrans(dict(zip(RUSSIAN, CLER)))


def transliterate(text):
    return text.translate(ENCODE).replace(" ", "<").replace("-", "<")


def check_digit(text):
    if not re.fullmatch(r"[A-Z0-9<]*", text):
        raise ValueError("Unsupported MRZ alphabet")
    values = [0 if c == "<" else int(c) if c.isdigit() else ord(c) - 55 for c in text]
    return str(sum(v * (7, 3, 1)[i % 3] for i, v in enumerate(values)) % 10)


class MRZ:
    def __init__(self, present=False, name_payload=""):
        self.present = present
        self.structure = False
        self.checks = {}
        self.values = {}
        self.name_payload = name_payload

    @property
    def trusted_numeric(self):
        return self.structure and all(self.checks.values())


def parse_mrz(value):
    result = MRZ(present=bool(value))
    if not isinstance(value, dict) or not isinstance(value.get("lines"), list):
        return result
    lines = value["lines"]
    if len(lines) != 2 or any(
        not isinstance(x, dict) or not isinstance(x.get("text"), str) for x in lines
    ):
        return result
    upper, lower = [x["text"].upper() for x in lines]
    if re.fullmatch(r"PNRUS[A-Z0-9<]{39}", upper):
        result.name_payload = upper[5:]
    result.structure = bool(re.fullmatch(r"[0-9]{10}RUS[0-9]{7}[MF]<{7}[0-9]{13}<[0-9]{2}", lower))
    if not result.structure:
        return result
    result.checks = {
        "number": check_digit(lower[:9]) == lower[9],
        "birth": check_digit(lower[13:19]) == lower[19],
        "optional": check_digit(lower[28:42]) == lower[42],
        "composite": check_digit(lower[:10] + lower[13:20] + lower[21:43]) == lower[43],
    }
    digits = lower[:3] + lower[28] + lower[3:9]
    result.values = {
        "Licence_number": f"{digits[:2]} {digits[2:4]} {digits[4:]}",
        "Birth_date_short": lower[13:19],
        "Sex_ru": "МУЖ." if lower[20] == "M" else "ЖЕН.",
        "Issue_date_short": lower[29:35],
        "Issue_organisation_code": lower[35:38] + "-" + lower[38:41],
    }
    return result


def names_agree(fields, mrz):
    expected = transliterate(fields["Last_name_ru"]) + "<<" + transliterate(fields["First_name_ru"])
    if fields["Middle_name_ru"]:
        expected += "<" + transliterate(fields["Middle_name_ru"])
    # Truncated MRZ cannot independently confirm a suffix or an absent patronymic.
    return len(expected) <= 39 and expected.ljust(39, "<") == mrz.name_payload

### Календарные и возрастные проверки


In [119]:
AS_OF = date(2026, 10, 1)


def parse_date(value):
    try:
        day, month, year = map(int, value.split("."))
        return date(year, month, day)
    except (ValueError, AttributeError, TypeError):
        return None


def anniversary(birth, years):
    """Feb 29 anniversaries use March 1; boundary cases are rejected below."""
    try:
        return birth.replace(year=birth.year + years)
    except ValueError:
        return date(birth.year + years, 3, 1)


def check_passport(fields, as_of=AS_OF):
    reasons = [f"format:{f}" for f in FIELDS if not valid_format(f, fields.get(f, ""))]
    birth = parse_date(fields.get("Birth_date", ""))
    issued = parse_date(fields.get("Issue_date", ""))
    if birth is None or issued is None:
        reasons.append("invalid_calendar_date")
    else:
        if birth > as_of or issued > as_of:
            reasons.append("future_date")
        if issued < date(1997, 10, 1):
            reasons.append("before_current_passport_introduction")
        if issued < anniversary(birth, 14):
            reasons.append("issued_before_14")
        for years in (20, 45):
            birthday = anniversary(birth, years)
            # Leap-day interpretation is immaterial except on the boundary itself.
            if (
                birth.month == 2
                and birth.day == 29
                and (abs((issued - birthday).days) <= 1 or abs((as_of - birthday).days - 90) <= 1)
            ):
                reasons.append("ambiguous_leap_day_boundary")
            if issued < birthday and as_of > birthday + timedelta(days=90):
                reasons.append(f"expired_at_{years}")
    return reasons

### Справочник и восстановление кандидата


In [120]:
def edit_distance(a, b):
    previous = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        current = [i]
        for j, cb in enumerate(b, 1):
            current.append(min(current[-1] + 1, previous[j] + 1, previous[j - 1] + (ca != cb)))
        previous = current
    return previous[-1]


class Subdivisions:
    def __init__(self, path):
        self.by_code = {}
        with path.open(encoding="utf-8-sig", newline="") as handle:
            for row in csv.DictReader(handle):
                self.by_code.setdefault(row["code"], set()).add(canonical(row["name"]))

    def match(self, code, text):
        names = self.by_code.get(code, set())
        if not names:
            return text, "unknown_code", 0
        if text in names:
            return text, "exact", 0
        if not text:
            return text, "missing_name", 0
        # Never infer a whole organization merely from its non-unique code.
        distances = sorted((edit_distance(text, name), name) for name in names)
        best, name = distances[0]
        runner_up = distances[1][0] if len(distances) > 1 else len(text)
        if best <= 2 and runner_up >= best + 2:
            return name, "unique_near", best
        return text, "ambiguous_or_distant", best


class Candidate:
    def __init__(self, fields, mrz):
        self.fields = fields
        self.mrz = mrz
        self.reasons = []
        self.changes = {}
        self.reference_status = "unchecked"
        self.reference_distance = 0
        self.name_agreement = False
        self.numeric_agreements = 0

    @property
    def eligible(self):
        return not self.reasons


def reconstruct(document, reference, corrected=True):
    viz = document.get("viz") or {}
    if not isinstance(viz, dict):
        viz = {}
    raw = {f: canonical(field_text(viz.get(f))) for f in FIELDS}
    fields = {f: normalize(f, raw[f]) if corrected else raw[f] for f in FIELDS}
    result = Candidate(fields, parse_mrz(document.get("mrz")))
    result.changes = {f: "normalization" for f in FIELDS if raw[f] != fields[f]}
    mrz = result.mrz
    if corrected and mrz.trusted_numeric:
        for f in ("Licence_number", "Issue_organisation_code", "Sex_ru"):
            target = mrz.values[f]
            if not fields[f]:
                fields[f] = target
                result.changes[f] = "mrz_missing"
            elif fields[f] != target:
                result.reasons.append(f"viz_mrz_conflict:{f}")
            else:
                result.numeric_agreements += 1
        for f, short_key in (
            ("Birth_date", "Birth_date_short"),
            ("Issue_date", "Issue_date_short"),
        ):
            parsed = parse_date(fields[f])
            if parsed and parsed.strftime("%y%m%d") == mrz.values[short_key]:
                result.numeric_agreements += 1
            elif f == "Issue_date" and not fields[f]:
                short = mrz.values[short_key]
                year = 2000 + int(short[:2])
                if year > AS_OF.year:
                    year -= 100
                fields[f] = f"{short[4:6]}.{short[2:4]}.{year:04d}"
                result.changes[f] = "mrz_missing"
            else:
                result.reasons.append(f"viz_mrz_conflict:{f}")
    result.name_agreement = names_agree(fields, mrz)
    if corrected:
        _, status, distance = reference.match(
            fields["Issue_organisation_code"], fields["Issue_organization_ru"]
        )
        result.reference_status, result.reference_distance = status, distance
        # A close reference spelling is evidence, not authority to rewrite the
        # printed wording: case endings and punctuation differ in ground truth.
    for f in FIELDS:
        if not fields[f] and f != "Middle_name_ru":
            result.reasons.append(f"missing:{f}")
    # A missing patronymic is accepted only if MRZ confirms the empty value.
    if not fields["Middle_name_ru"] and not result.name_agreement:
        result.reasons.append("unconfirmed_empty_patronymic")
    result.reasons.extend(check_passport(fields))
    return result

### Признаки и обязательные проверки


In [121]:
def valid_confidence(value):
    return type(value) in (int, float) and 0 <= value <= 1


def confidence_values(value):
    if not value:
        return []
    return [
        float(number)
        for line in value.get("lines", [])
        for number in line.get("char_conf", []) or []
        if valid_confidence(number)
    ]


def geometry(value):
    """Largest unobserved vertical gap, relative to a recognized line height."""
    if not value or not value.get("lines"):
        return 1.0, 0.0, 0
    field_box = value["box"]
    boxes = [line["box"] for line in value["lines"]]
    if any(
        len(box) != 4
        or not all(isinstance(x, (int, float)) and math.isfinite(x) for x in box)
        or box[3] <= 0
        for box in [field_box, *boxes]
    ):
        return 1.0, 0.0, 0
    boxes.sort(key=lambda box: box[1])
    line_height = mean(box[3] for box in boxes)
    gaps = [boxes[0][1] - field_box[1], field_box[1] + field_box[3] - boxes[-1][1] - boxes[-1][3]]
    gaps.extend(b[1] - a[1] - a[3] for a, b in zip(boxes, boxes[1:]))
    return (
        max(0.0, max(gaps)) / line_height,
        sum(box[3] for box in boxes) / field_box[3],
        len(boxes),
    )


def features(document, candidate):
    result = {}
    viz = document.get("viz") or {}
    for f in FIELDS:
        value = viz.get(f)
        chars = sorted(confidence_values(value))
        conf = value.get("conf", 0) if isinstance(value, dict) else 0
        conf = float(conf) if valid_confidence(conf) else 0.0
        prefix = f + ":"
        result[prefix + "conf"] = conf
        result[prefix + "min"] = chars[0] if chars else 0
        result[prefix + "mean"] = mean(chars) if chars else 0
        result[prefix + "q10"] = chars[int((len(chars) - 1) * 0.1)] if chars else 0
        result[prefix + "below98"] = sum(x < 0.98 for x in chars)
        result[prefix + "below90"] = sum(x < 0.9 for x in chars)
        result[prefix + "missing_chars"] = float(not chars)
        result[prefix + "length"] = len(candidate.fields[f])
        result[prefix + "changed"] = float(f in candidate.changes)
        gap, coverage, line_count = geometry(value)
        result[prefix + "geometry_gap"] = gap
        result[prefix + "geometry_coverage"] = coverage
        result[prefix + "line_count"] = line_count
    for source in ("web", "mobile", "mobile_v2"):
        result["source:" + source] = float(document.get("source") == source)
    for name in ("sharpness", "doc_px", "glare"):
        value = (document.get("quality") or {}).get(name)
        result["quality:" + name] = (
            float(value) if isinstance(value, (int, float)) and math.isfinite(value) else 0
        )
    result["mrz:present"] = float(candidate.mrz.present)
    result["mrz:structure"] = float(candidate.mrz.structure)
    result["mrz:checks"] = sum(candidate.mrz.checks.values())
    result["mrz:names_agree"] = float(candidate.name_agreement)
    result["mrz:numeric_agreements"] = candidate.numeric_agreements
    mrz_chars = confidence_values(document.get("mrz"))
    result["mrz:min"] = min(mrz_chars, default=0)
    result["mrz:mean"] = mean(mrz_chars) if mrz_chars else 0
    result["reference:exact"] = float(candidate.reference_status == "exact")
    result["reference:near"] = float(candidate.reference_status == "unique_near")
    result["reference:distance"] = candidate.reference_distance
    result["changes"] = len(candidate.changes)
    all_chars = [x for v in viz.values() for x in confidence_values(v)]
    result["saturated_confidence"] = float(bool(all_chars) and min(all_chars) == 1)
    return result


def evidence_gate(document, candidate):
    """Reject missing lines and sources whose OCR confidence carries no variation."""
    viz = document.get("viz") or {}
    all_chars = [x for value in viz.values() for x in confidence_values(value)]
    if not all_chars or min(all_chars) == 1:
        return False
    # MRZ does not resolve the century of birth. Supported centuries are policy.
    if candidate.fields["Birth_date"][-4:-2] not in {"19", "20"}:
        return False
    for field in FIELDS:
        if not candidate.fields[field] or candidate.changes.get(field) == "mrz_missing":
            continue
        value = viz.get(field)
        chars = confidence_values(value)
        if not chars:
            return False
        lines = value["lines"]
        if any(len(line["text"]) != len(line.get("char_conf") or []) for line in lines):
            return False
        if len(chars) != sum(len(line["text"]) for line in lines):
            return False
        if geometry(value)[0] > 0.6:
            return False
    return True


def rule_score(document, candidate, strict):
    """Joint evidence gate plus the weakest unverified character confidence."""
    if not candidate.eligible:
        return -1.0
    if strict and not (candidate.mrz.trusted_numeric and candidate.name_agreement):
        return -1.0
    viz = document.get("viz") or {}
    checked = ("Birth_place_ru", "Issue_organization_ru") if strict else FIELDS
    scores = []
    for f in checked:
        if not candidate.fields[f] and f == "Middle_name_ru":
            continue
        chars = confidence_values(viz.get(f))
        scores.append(min(chars, default=0))
    return min(scores, default=0)

### Загрузка данных и метрики


In [122]:
import html
import io
import unittest

from IPython.display import HTML, Markdown, display


def read_jsonl(path):
    with path.open(encoding="utf-8") as handle:
        return [json.loads(line) for line in handle]


def is_correct(fields, label):
    return all(
        canonical(fields.get(f)) == canonical(label[f]) for f in FIELDS
    ) and not check_passport(label)


def binomial_upper(errors, accepted, alpha=0.05):
    """Exact one-sided Clopper-Pearson bound, including the zero-error case."""
    if accepted == 0:
        return None
    if errors == 0:
        return -math.expm1(math.log(alpha) / accepted)
    if errors == accepted:
        return 1.0
    from scipy.stats import beta

    return float(beta.ppf(1 - alpha, errors + 1, accepted - errors))


def metrics(correct, accepts):
    count = sum(accepts)
    wrong = sum(a and not c for a, c in zip(accepts, correct))
    return {
        "N_total": len(correct),
        "N_accepted": count,
        "N_rejected": len(correct) - count,
        "N_correct_accepted": count - wrong,
        "N_wrong_accepted": wrong,
        "acceptance_rate": count / len(correct) if correct else 0,
        "false_acceptance_rate": wrong / count if count else None,
        "precision_among_accepted": (count - wrong) / count if count else None,
        "error_upper_95_one_sided": binomial_upper(wrong, count),
        "rule_of_three": min(1, 3 / count) if count and wrong == 0 else None,
    }


def table(rows, columns=None):
    rows = list(rows)
    if not rows:
        display(Markdown("_Нет строк._"))
        return
    columns = columns or list(rows[0])

    def render_row(values, tag):
        cells = "".join(f"<{tag}>{html.escape(str(v))}</{tag}>" for v in values)
        return f"<tr>{cells}</tr>"

    header = render_row(columns, "th")
    body = "".join(render_row((row.get(c, "") for c in columns), "td") for row in rows)
    markup = f"<thead>{header}</thead><tbody>{body}</tbody>"
    display(HTML(f'<div style="overflow-x:auto"><table>{markup}</table></div>'))


def prepare_documents(documents, reference):
    candidates = [reconstruct(doc, reference) for doc in documents]
    rows = [features(doc, candidate) for doc, candidate in zip(documents, candidates)]
    return candidates, rows


def run_tests(cases):
    suite = unittest.TestSuite(
        unittest.defaultTestLoader.loadTestsFromTestCase(case) for case in cases
    )
    stream = io.StringIO()
    result = unittest.TextTestRunner(stream=stream, verbosity=2).run(suite)
    print(stream.getvalue())
    assert result.wasSuccessful()
    table(
        [
            {
                "Выполнено тестов": result.testsRun,
                "Ошибок": len(result.errors),
                "Провалов": len(result.failures),
            }
        ]
    )
    return result

In [123]:
import copy
from collections import Counter

# Notebook запускается из каталога code.
PROJECT_ROOT = Path.cwd()

## 1. Данные

Вход — строки OCR с визуальными полями, MRZ и confidence. Для оценки восстановлений используются train-метки.


In [124]:
print(PROJECT_ROOT)
train_documents = read_jsonl(PROJECT_ROOT / "data/train_predictions.jsonl")
test_documents = read_jsonl(PROJECT_ROOT / "data/test_predictions.jsonl")
labels = {
    row["doc_id"]: row["fields"] for row in read_jsonl(PROJECT_ROOT / "data/train_labels.jsonl")
}
reference = Subdivisions(PROJECT_ROOT / "reference/subdivisions.csv")
print(f"Загружено документов: train={len(train_documents)}, test={len(test_documents)}.")

/Users/daniil/Downloads/candidate
Загружено документов: train=3000, test=2000.


## 2. Нормализация полей

Числовые омоглифы исправляются в числовых реквизитах. В свободном тексте сохраняются пунктуация, сокращения, `Ё` и иностранные названия. Нормализация не должна произвольно менять правильное значение.


In [125]:
normalization_rows = []
normalization_examples = []
for field in FIELDS:
    changed = fixed = damaged = 0
    for doc in train_documents:
        raw = canonical(field_text(doc["viz"].get(field)))
        normalized = normalize(field, raw)
        gold = canonical(labels[doc["doc_id"]][field])
        if raw != normalized:
            changed += 1
            fixed += raw != gold and normalized == gold
            damaged += raw == gold and normalized != gold
            if len(normalization_examples) < 8 and field in {"Sex_ru", "Issue_organisation_code"}:
                normalization_examples.append(
                    {"Поле": field, "OCR": raw, "Нормализовано": normalized, "Метка": gold}
                )
    normalization_rows.append(
        {"Поле": field, "Изменено": changed, "Исправлено": fixed, "Испорчено": damaged}
    )
table(normalization_rows)
table(normalization_examples)

Поле,Изменено,Исправлено,Испорчено
Last_name_ru,0,0,0
First_name_ru,0,0,0
Middle_name_ru,0,0,0
Sex_ru,3,3,0
Birth_date,0,0,0
Birth_place_ru,0,0,0
Issue_date,0,0,0
Issue_organisation_code,13,12,0
Licence_number,3,0,0
Issue_organization_ru,0,0,0


Поле,OCR,Нормализовано,Метка
Sex_ru,ЖЕН,ЖЕН.,ЖЕН.
Sex_ru,ЖЕН,ЖЕН.,ЖЕН.
Sex_ru,ЖЕН,ЖЕН.,ЖЕН.
Issue_organisation_code,640041,640-041,640-041
Issue_organisation_code,700026,700-026,700-026
Issue_organisation_code,770037,770-037,770-037
Issue_organisation_code,230025,230-025,230-025
Issue_organisation_code,670025,670-025,670-025


## 3. Разбор MRZ

MRZ внутреннего паспорта — две строки по 44 символа. Разбор учитывает модернизированный клер, отдельную четвёртую цифру серии, дату выдачи и код подразделения. Проверяются четыре числовые контрольные цифры.

Пол и ФИО этими checksum не защищены. Место рождения и название органа выдачи в MRZ не записаны. Согласие VIZ/MRZ не исключает общей ошибки двух зон.


In [126]:
mrz_checks = Counter()
for doc in train_documents:
    parsed = parse_mrz(doc.get("mrz"))
    mrz_checks["MRZ обнаружена"] += parsed.present
    mrz_checks["Структура нижней строки корректна"] += parsed.structure
    mrz_checks["Все числовые checksum пройдены"] += parsed.trusted_numeric
    mrz_checks["Полное ФИО совпадает с меткой"] += names_agree(labels[doc["doc_id"]], parsed)
table([{"Проверка MRZ": name, "Документов": count} for name, count in mrz_checks.items()])

Проверка MRZ,Документов
MRZ обнаружена,2817
Структура нижней строки корректна,2543
Все числовые checksum пройдены,2519
Полное ФИО совпадает с меткой,2454


## 4. Восстановление полей из MRZ

Отсутствующие реквизиты визуальной зоны заполняются значениями из MRZ того же документа при прохождении числовых контрольных проверок. Для каждого изменения сохраняется источник восстановления.

| Поле | Разрешённый источник восстановления | Ограничение |
|---|---|---|
| Серия и номер | Проверенная числовая MRZ | Четвёртая цифра серии берётся из дополнительных данных |
| Код подразделения | Проверенная числовая MRZ | Проходят проверки дополнительных данных и составная проверка |
| Дата выдачи | Проверенная числовая MRZ | Век выбирается в диапазоне выпуска текущего паспорта; календарь/будущая дата проверяются |
| Пол | Символ M/F в MRZ | **Сам символ пола не защищён checksum**; это более слабое восстановление |
| Дата рождения | Не восстанавливается при пропуске | Двузначный год не позволяет надёжно выбрать век |
| Место рождения / орган выдачи | Не восстанавливаются | MRZ этих данных не содержит |

Противоречия между заполненными полями VIZ и MRZ записываются как причины отклонения. Подготовленные значения проходят остальные проверки документа. Влияние восстановления на покрытие оценивается в `analysis.ipynb` после обучения модели.


In [127]:
# Реальная реконструкция; метки нужны только для оценки, reconstruct их не получает.
train_candidates, train_features = prepare_documents(train_documents, reference)
test_candidates, test_features = prepare_documents(test_documents, reference)
train_recovered = [
    [f for f, mode in c.changes.items() if mode == "mrz_missing"] for c in train_candidates
]
test_recovered = [
    [f for f, mode in c.changes.items() if mode == "mrz_missing"] for c in test_candidates
]
recovery_rows = []
for field in ["Licence_number", "Issue_organisation_code", "Issue_date", "Sex_ru"]:
    indexes = [i for i, recovered in enumerate(train_recovered) if field in recovered]
    wrong = sum(
        canonical(train_candidates[i].fields[field])
        != canonical(labels[train_documents[i]["doc_id"]][field])
        for i in indexes
    )
    recovery_rows.append(
        {
            "Поле": field,
            "Восстановлено train": len(indexes),
            "Ошибок поля на train": wrong,
            "Восстановлено test": sum(field in recovered for recovered in test_recovered),
        }
    )
table(recovery_rows)
print(
    "Документов с хотя бы одним восстановлением:",
    sum(map(bool, train_recovered)),
    "train;",
    sum(map(bool, test_recovered)),
    "test",
)
print(
    "Число восстановленных полей может превышать число документов: в документе иногда несколько пропусков."
)

Поле,Восстановлено train,Ошибок поля на train,Восстановлено test
Licence_number,25,0,19
Issue_organisation_code,22,0,14
Issue_date,11,0,9
Sex_ru,20,0,15


Документов с хотя бы одним восстановлением: 77 train; 56 test
Число восстановленных полей может превышать число документов: в документе иногда несколько пропусков.


In [128]:
# Примеры восстановленных значений; окончательное решение о принятии будет после обучения.
table(
    [
        {
            "doc_id": doc["doc_id"],
            "Поле": field,
            "OCR в визуальной зоне": field_text(doc["viz"].get(field)) or "поле отсутствует",
            "Восстановлено из MRZ": candidate.fields[field],
            "Эталон train": labels[doc["doc_id"]][field],
        }
        for doc, candidate, recovered in zip(train_documents, train_candidates, train_recovered)
        for field in recovered
    ][:5]
)

doc_id,Поле,OCR в визуальной зоне,Восстановлено из MRZ,Эталон train
tr_00010,Issue_organisation_code,поле отсутствует,380-046,380-046
tr_00150,Sex_ru,поле отсутствует,ЖЕН.,ЖЕН.
tr_00153,Issue_organisation_code,поле отсутствует,770-102,770-102
tr_00179,Issue_organisation_code,поле отсутствует,170-001,170-001
tr_00187,Issue_organisation_code,поле отсутствует,630-037,630-037


Качество восстановленных значений оценивается сравнением с train-разметкой. Пол в MRZ не покрывается числовыми контрольными суммами; его восстановление использует более слабое подтверждение, чем восстановление номера или кода подразделения.


## 5. Геометрия строк и даты

Вертикальные зазоры между рамкой поля и распознанными строками используются для обнаружения пропущенных строк. Проверки дат включают календарную корректность, возраст при выдаче и возрастные границы 20 и 45 лет с интервалом 90 дней. Расчёт выполняется на 01.10.2026.


In [129]:
geometry_rows = []
for doc in train_documents:
    for field in ("Birth_place_ru", "Issue_organization_ru"):
        value = doc["viz"].get(field)
        gap, coverage, line_count = geometry(value)
        if value and gap > 0.6:
            geometry_rows.append(
                {
                    "doc_id": doc["doc_id"],
                    "Поле": field,
                    "Строк": line_count,
                    "Относительный зазор": round(gap, 3),
                    "Покрытие рамки": round(coverage, 3),
                }
            )
table(geometry_rows[:8])
validity_reasons = Counter(
    reason for fields_value in labels.values() for reason in check_passport(fields_value)
)
table(
    [
        {"Причина отклонения эталона по офлайн-проверкам": reason, "Документов": count}
        for reason, count in validity_reasons.items()
    ]
)

doc_id,Поле,Строк,Относительный зазор,Покрытие рамки
tr_00019,Birth_place_ru,1,0.96,0.5
tr_00039,Issue_organization_ru,2,1.038,0.65
tr_00059,Birth_place_ru,1,1.094,0.478
tr_00104,Issue_organization_ru,2,1.037,0.675
tr_00165,Issue_organization_ru,1,0.966,0.492
tr_00248,Birth_place_ru,1,1.059,0.486
tr_00382,Birth_place_ru,1,1.083,0.48
tr_00387,Issue_organization_ru,2,1.072,0.651


Причина отклонения эталона по офлайн-проверкам,Документов
expired_at_20,78
expired_at_45,12


## 6. Числовые признаки для обучения

Каждый документ превращается в набор чисел: уверенность OCR, геометрия строк, результаты MRZ и другие признаки качества. Эти значения будут входом модели во втором notebook.


In [130]:
feature_names = list(train_features[0])
print(
    "Подготовлено:",
    len(train_features),
    "train и",
    len(test_features),
    "test; признаков на документ:",
    len(feature_names),
)
table(
    [
        {
            "doc_id": doc["doc_id"],
            "Мин. confidence места рождения": values["Birth_place_ru:min"],
            "Проверок MRZ пройдено": values["mrz:checks"],
            "Изменено полей": values["changes"],
        }
        for doc, values in zip(train_documents[:5], train_features[:5])
    ]
)

Подготовлено: 3000 train и 2000 test; признаков на документ: 138


doc_id,Мин. confidence места рождения,Проверок MRZ пройдено,Изменено полей
tr_00001,0.9971,4,0
tr_00002,0.5398,0,0
tr_00003,0.9927,4,0
tr_00004,1.0,4,0
tr_00005,0.9929,4,0


## 7. Тесты предобработки

22 теста проверяют нормализацию, MRZ, даты, восстановление, геометрию и статистические функции. Они выполняются до обучения и не используют сохранённую модель.

Ещё 6 тестов — применение модели через CLI и корректность разбиений — находятся в конце `analysis.ipynb`, после обучения.


In [131]:
def fields():
    return {
        "Last_name_ru": "ИВАНОВ",
        "First_name_ru": "ИВАН",
        "Middle_name_ru": "ИВАНОВИЧ",
        "Sex_ru": "МУЖ.",
        "Birth_date": "01.01.2000",
        "Birth_place_ru": "Г. МОСКВА",
        "Issue_date": "01.01.2021",
        "Issue_organisation_code": "770-001",
        "Licence_number": "45 21 123456",
        "Issue_organization_ru": "ГУ МВД РОССИИ ПО Г. МОСКВЕ",
    }


def mrz_for(values):
    def digit(text):
        return str(
            sum((int(c) if c != "<" else 0) * (7, 3, 1)[i % 3] for i, c in enumerate(text)) % 10
        )

    names = transliterate(values["Last_name_ru"]) + "<<" + transliterate(values["First_name_ru"])
    if values["Middle_name_ru"]:
        names += "<" + transliterate(values["Middle_name_ru"])
    number = values["Licence_number"].replace(" ", "")
    head = number[:3] + number[4:]
    birth = parse_date(values["Birth_date"]).strftime("%y%m%d")
    issued = parse_date(values["Issue_date"]).strftime("%y%m%d")
    optional = number[3] + issued + values["Issue_organisation_code"].replace("-", "") + "<"
    sex = "M" if values["Sex_ru"] == "МУЖ." else "F"
    lower = f"{head}{digit(head)}RUS{birth}{digit(birth)}{sex}<<<<<<<{optional}{digit(optional)}"
    lower += digit(lower[:10] + lower[13:20] + lower[21:43])
    return {
        "lines": [
            {"text": "PNRUS" + names.ljust(39, "<"), "char_conf": [0.999] * 44},
            {"text": lower, "char_conf": [0.999] * 44},
        ]
    }


def document():
    values = fields()
    return {
        "doc_id": "unit-fixture",
        "source": "web",
        "quality": {"sharpness": 200, "doc_px": 1800, "glare": 0.01},
        "viz": {
            f: {
                "conf": 0.999,
                "box": [0, 0, 500, 30],
                "lines": [
                    {"text": value, "box": [0, 1, 490, 28], "char_conf": [0.999] * len(value)}
                ],
            }
            for f, value in values.items()
        },
        "mrz": mrz_for(values),
    }

### Нормализация


In [132]:
class NormalizationTests(unittest.TestCase):
    def test_every_field(self):
        cases = {
            "Last_name_ru": (" ивAнов ", "ИВАНОВ"),
            "First_name_ru": ("  иван ", "ИВАН"),
            "Middle_name_ru": ("МУРЗАФАР-ОГЛЫ", "МУРЗАФАР-ОГЛЫ"),
            "Sex_ru": ("муж", "МУЖ."),
            "Birth_date": ("01/02/2000", "01.02.2000"),
            "Birth_place_ru": (" Г.  МОСКВА\n", "Г. МОСКВА"),
            "Issue_date": ("01-02-2021", "01.02.2021"),
            "Issue_organisation_code": ("77ООО1", "770-001"),
            "Licence_number": ("45 21 12З456", "45 21 123456"),
            "Issue_organization_ru": ("ГУ МВД  РОCCИИ", "ГУ МВД РОССИИ"),
        }
        self.assertEqual(set(cases), set(FIELDS))
        for f, (raw, expected) in cases.items():
            with self.subTest(field=f):
                self.assertEqual(normalize(f, raw), expected)

    def test_context_and_unicode(self):
        cases = [
            ("Last_name_ru", "СОКОЛОВ"),
            ("Birth_place_ru", "Г. SVERDLOVS'K УКРАИНСКАЯ ССР"),
            ("Issue_organization_ru", "ТП №1 ОУФМС"),
            ("Last_name_ru", "ЗВЕРЕВ"),
            ("Last_name_ru", "СЁМИН"),
        ]
        for field, value in cases:
            with self.subTest(field=field, value=value):
                self.assertEqual(normalize(field, value), value)
        self.assertFalse(valid_format("Last_name_ru", "ИВ?НОВ"))

    def test_no_arbitrary_abbreviation_rewrite(self):
        for value in (
            "Г.МОСКВА",
            "Г. МОСКВА",
            "ГУ МВД РОССИИ ПО САРАТОВСКОЙ ОБЛ",
            "ОТДЕЛЕНИЕ УФМС",
        ):
            self.assertEqual(normalize("Issue_organization_ru", value), value)

### MRZ и контрольные цифры


In [133]:
class MRZTests(unittest.TestCase):
    def test_sex_is_not_protected_by_numeric_checksums(self):
        value = mrz_for(fields())
        lower = value["lines"][1]["text"]
        self.assertEqual(lower[20], "M")
        value["lines"][1]["text"] = lower[:20] + "F" + lower[21:]
        result = parse_mrz(value)
        self.assertTrue(result.trusted_numeric)
        self.assertEqual(result.values["Sex_ru"], "ЖЕН.")

    def test_known_checksum_and_internal_layout(self):
        self.assertEqual(check_digit("123456789"), "7")
        self.assertEqual(check_digit("<<<<<<"), "0")
        lines = [
            "PNRUSZAINTINOV<<ZAKERIQ<NAZIFULOVI3<<<<<<<<<",
            "6122839687RUS1006102M<<<<<<<4250911620004<28",
        ]
        result = parse_mrz({"lines": [{"text": text} for text in lines]})
        self.assertTrue(result.trusted_numeric)
        self.assertEqual(result.values["Licence_number"], "61 24 283968")
        self.assertEqual(result.values["Issue_date_short"], "250911")
        self.assertEqual(result.values["Issue_organisation_code"], "620-004")

    def test_invalid_mrz_not_repaired_by_checksum_guess(self):
        value = mrz_for(fields())
        text = value["lines"][1]["text"]
        value["lines"][1]["text"] = ("9" if text[0] != "9" else "8") + text[1:]
        self.assertFalse(parse_mrz(value).trusted_numeric)
        for bad in (
            None,
            {},
            {"lines": []},
            {"lines": [{"text": "PNRUS"}]},
            {"lines": [{"text": 3}]},
        ):
            self.assertFalse(parse_mrz(bad).trusted_numeric)

    def test_transliteration_and_name_ambiguity(self):
        self.assertEqual(transliterate("ЁЖИЙЧШЩЪЫЬЭЮЯ"), "2JIQ34WXY9678")
        values = fields()
        self.assertTrue(names_agree(values, parse_mrz(mrz_for(values))))
        values["Last_name_ru"] = "А" * 40
        self.assertFalse(names_agree(values, MRZ(name_payload="А" * 39)))

    def test_check_digits_are_not_independent_name_proof(self):
        values = fields()
        value = mrz_for(values)
        value["lines"][0]["text"] = value["lines"][0]["text"].replace("IVANOV", "IVANOF", 1)
        result = parse_mrz(value)
        self.assertTrue(result.trusted_numeric)
        self.assertFalse(names_agree(values, result))

### Календарные и возрастные проверки


In [134]:
class ValidationTests(unittest.TestCase):
    def test_dates(self):
        self.assertIsNone(parse_date("31.02.2020"))
        self.assertIsNone(parse_date("not a date"))
        self.assertEqual(parse_date("29.02.2000"), date(2000, 2, 29))
        self.assertEqual(anniversary(date(2000, 2, 29), 45), date(2045, 3, 1))

    def test_age14_and_future(self):
        values = fields()
        values["Issue_date"] = "31.12.2013"
        self.assertIn("issued_before_14", check_passport(values))
        values["Issue_date"] = "02.10.2026"
        self.assertIn("future_date", check_passport(values))
        values["Birth_date"] = "01.01.2027"
        self.assertTrue(check_passport(values))

    def test_20_and_45_replacement_inclusive_90_days(self):
        for birth_year, years, issue_year in ((2000, 20, 2015), (1975, 45, 2000)):
            values = fields()
            values["Birth_date"] = f"01.01.{birth_year}"
            values["Issue_date"] = f"01.01.{issue_year}"
            birthday = date(birth_year + years, 1, 1)
            self.assertFalse(check_passport(values, birthday + timedelta(days=90)))
            self.assertIn(
                f"expired_at_{years}", check_passport(values, birthday + timedelta(days=91))
            )
            values["Issue_date"] = birthday.strftime("%d.%m.%Y")
            self.assertFalse(check_passport(values, birthday + timedelta(days=91)))

    def test_valid_fields_and_number_format(self):
        self.assertEqual(check_passport(fields()), [])
        values = fields()
        values["Licence_number"] = "1234"
        self.assertTrue(check_passport(values))

### Восстановление и повреждённые поля


In [135]:
class ReconstructionTests(unittest.TestCase):
    @classmethod
    def setUpClass(cls):
        cls.reference = Subdivisions(PROJECT_ROOT / "reference/subdivisions.csv")

    def test_missing_fields_and_mrz(self):
        doc = document()
        for f in ("Sex_ru", "Licence_number", "Issue_date", "Issue_organisation_code"):
            doc["viz"][f] = None
        candidate = reconstruct(doc, self.reference)
        self.assertTrue(candidate.eligible)
        self.assertEqual(candidate.fields, fields())
        doc["mrz"] = None
        self.assertFalse(reconstruct(doc, self.reference).eligible)

    def test_no_guessed_birth_century(self):
        doc = document()
        doc["viz"]["Birth_date"] = None
        self.assertFalse(reconstruct(doc, self.reference).eligible)

    def test_empty_visual_zone(self):
        self.assertFalse(reconstruct({"viz": None}, self.reference).eligible)

    def test_conflict_and_corruption(self):
        doc = document()
        doc["viz"]["Licence_number"]["lines"][0]["text"] = "45 21 123457"
        self.assertIn("viz_mrz_conflict:Licence_number", reconstruct(doc, self.reference).reasons)
        doc["viz"]["Birth_date"]["lines"][0]["text"] = "99.99.9999"
        self.assertFalse(reconstruct(doc, self.reference).eligible)

    def test_missing_patronymic_requires_confirmation(self):
        doc = document()
        doc["viz"]["Middle_name_ru"] = None
        self.assertFalse(reconstruct(doc, self.reference).eligible)
        values = fields()
        values["Middle_name_ru"] = ""
        doc["mrz"] = mrz_for(values)
        self.assertTrue(reconstruct(doc, self.reference).eligible)

    def test_reference_ambiguity_does_not_change_value(self):
        ref = copy.deepcopy(self.reference)
        ref.by_code = {"770-001": {"АБВ", "АБГ"}}
        self.assertEqual(ref.match("770-001", "АБД")[1], "ambiguous_or_distant")
        self.assertEqual(ref.match("999-999", "АБД")[1], "unknown_code")
        doc = document()
        ref.by_code = {"770-001": {fields()["Issue_organization_ru"] + "."}}
        self.assertEqual(
            reconstruct(doc, ref).fields["Issue_organization_ru"], fields()["Issue_organization_ru"]
        )

    def test_geometry_detects_lost_first_middle_and_last_line(self):
        value = {"box": [0, 0, 100, 90], "lines": [{"box": [0, y, 100, 29]} for y in (0, 30, 60)]}
        self.assertLess(geometry(value)[0], 0.6)
        for index in range(3):
            partial = copy.deepcopy(value)
            del partial["lines"][index]
            self.assertGreater(geometry(partial)[0], 0.6)

    def test_missing_and_saturated_confidence(self):
        self.assertEqual(
            confidence_values({"lines": [{"char_conf": [None, "bad", float("nan"), -1, 2]}]}), []
        )
        doc = document()
        for v in doc["viz"].values():
            for line in v["lines"]:
                line["char_conf"] = [1.0] * len(line["text"])
        self.assertFalse(evidence_gate(doc, reconstruct(doc, self.reference)))
        doc = document()
        doc["viz"]["Birth_place_ru"]["lines"][0]["char_conf"][0] = None
        self.assertFalse(evidence_gate(doc, reconstruct(doc, self.reference)))

### Статистические метрики


In [136]:
class StatisticsTests(unittest.TestCase):
    def test_no_accepts_is_undefined_not_zero_risk(self):
        result = metrics([True], [False])
        self.assertIsNone(result["false_acceptance_rate"])
        self.assertIsNone(result["error_upper_95_one_sided"])

    def test_exact_zero_error_bound(self):
        self.assertAlmostEqual(binomial_upper(0, 100), 1 - 0.05**0.01)
        self.assertEqual(binomial_upper(10, 10), 1)

In [137]:
test_result = run_tests(
    [
        NormalizationTests,
        MRZTests,
        ValidationTests,
        ReconstructionTests,
        StatisticsTests,
    ]
)

test_context_and_unicode (__main__.NormalizationTests.test_context_and_unicode) ... ok
test_every_field (__main__.NormalizationTests.test_every_field) ... ok
test_no_arbitrary_abbreviation_rewrite (__main__.NormalizationTests.test_no_arbitrary_abbreviation_rewrite) ... ok
test_check_digits_are_not_independent_name_proof (__main__.MRZTests.test_check_digits_are_not_independent_name_proof) ... ok
test_invalid_mrz_not_repaired_by_checksum_guess (__main__.MRZTests.test_invalid_mrz_not_repaired_by_checksum_guess) ... ok
test_known_checksum_and_internal_layout (__main__.MRZTests.test_known_checksum_and_internal_layout) ... ok
test_sex_is_not_protected_by_numeric_checksums (__main__.MRZTests.test_sex_is_not_protected_by_numeric_checksums) ... ok
test_transliteration_and_name_ambiguity (__main__.MRZTests.test_transliteration_and_name_ambiguity) ... ok
test_20_and_45_replacement_inclusive_90_days (__main__.ValidationTests.test_20_and_45_replacement_inclusive_90_days) ... ok
test_age14_and_futur

Выполнено тестов,Ошибок,Провалов
22,0,0


## Результат

Исходные OCR-поля подготовлены, MRZ и даты проверены, восстановления показаны на реальных документах. Функции `reconstruct()` и `features()` используются при обучении и обработке новых документов.

Далее откройте [analysis.ipynb](analysis.ipynb) и выполните `Run All`. Он обучит модель, сохранит её, сформирует ответы и выполнит тесты CLI.
